# 04 — Phân lớp có giám sát (Classification)

Mục tiêu:
- Huấn luyện Random Forest & XGBoost
- Đánh giá: F1 (macro), PR-AUC
- Giải thích mô hình bằng SHAP
- **So sánh bảng + biểu đồ**

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from src.data.loader import load_data, load_config
from src.data.cleaner import HRDataCleaner
from src.features.builder import FeatureBuilder
from src.models.supervised import train_random_forest, train_xgboost
from src.evaluation.metrics import evaluate_classifier, explain_with_shap, build_comparison_table
from src.visualization.plots import plot_model_comparison, plot_feature_importance_top10

config = load_config("configs/params.yaml")

## 4.1 Chuẩn bị dữ liệu

**Experimental Setup:**
- Train/Test split: 80/20 (stratified)
- Random seed: 42
- Class balance: class_weight='balanced' (RF), scale_pos_weight (XGB)

In [ ]:
df = load_data(config["data"]["raw_path"])
cleaner = HRDataCleaner(target_col="Attrition")
df_clean = cleaner.clean(df)

# Feature engineering
fb = FeatureBuilder(df_clean)
df_featured = fb.build_all()
df_encoded = cleaner.encode(df_featured)

X = df_encoded.drop(columns=["Attrition"])
y = df_encoded["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config["data"]["test_size"],
    random_state=config["model"]["random_state"], stratify=y,
)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"Attrition rate (train): {y_train.mean():.1%}")

## 4.2 Random Forest

In [ ]:
rf_model = train_random_forest(
    X_train, y_train,
    n_estimators=config["model"]["rf_n_estimators"],
    random_state=config["model"]["random_state"],
)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]
rf_results = evaluate_classifier(y_test, rf_pred, rf_prob, model_name="Random Forest")
rf_results["model_name"] = "Random Forest"

## 4.3 XGBoost

In [ ]:
xgb_model = train_xgboost(
    X_train, y_train,
    n_estimators=config["model"]["xgb_n_estimators"],
    learning_rate=config["model"]["xgb_learning_rate"],
    max_depth=config["model"]["xgb_max_depth"],
    random_state=config["model"]["random_state"],
)
xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
xgb_results = evaluate_classifier(y_test, xgb_pred, xgb_prob, model_name="XGBoost")
xgb_results["model_name"] = "XGBoost"

## 4.4 So sánh RF vs XGBoost (Bảng + Biểu đồ)

In [ ]:
comparison = build_comparison_table([rf_results, xgb_results])
plot_model_comparison(comparison)

## 4.5 Giải thích SHAP — Top 10 Features

> **Global feature importance**: các features nào ảnh hưởng nhiều nhất đến dự đoán nghỉ việc?

In [ ]:
# SHAP cho mô hình tốt nhất
best_name = comparison.loc[comparison["F1 (macro)"].idxmax(), "Model"]
print(f"Mô hình tốt nhất: {best_name}")

if "Random Forest" in best_name:
    shap_values, feature_imp = explain_with_shap(rf_model, X_test)
else:
    shap_values, feature_imp = explain_with_shap(xgb_model, X_test)

if feature_imp is not None:
    plot_feature_importance_top10(feature_imp)

## 4.6 Nhận xét

**So sánh RF vs XGBoost:**
- Cả hai đều xử lý tốt dữ liệu mất cân bằng nhờ class_weight/scale_pos_weight
- PR-AUC phù hợp hơn accuracy cho đánh giá

**Top features (SHAP):**
- OverTime, MonthlyIncome, Age là yếu tố quyết định
- Các feature phái sinh (WorkloadScore, SatisfactionIndex) cũng đóng góp

→ Kết quả này nhất quán với EDA ở notebook 01.